A latent-variable model defines the density of an observation $\mathbf{x}$ only through an integral over the latent variable $\mathbf{z}$:

$$
p(\mathbf{x}) = \int p(\mathbf{x} \mid \mathbf{z})\, p(\mathbf{z})\, d\mathbf{z} = \mathbb{E}_{p(\mathbf{z})}\big[p(\mathbf{x} \mid \mathbf{z})\big].
$$

The expectation form suggests the simplest estimator. Draw $\mathbf{z}_1, \dots, \mathbf{z}_N$ from the prior and average the likelihoods, $\hat{p}(\mathbf{x}) = \frac{1}{N} \sum_{i} p(\mathbf{x} \mid \mathbf{z}_i)$. This is *naive Monte Carlo*. It is unbiased, but almost all of the integral comes from the small region of latent space that explains $\mathbf{x}$ well. Prior samples rarely land there, and the region shrinks, relative to the prior, as the latent dimension grows.

To measure this failure you need a model where $p(\mathbf{x})$ is known exactly. Then every estimate can be compared to the true value, and the error belongs to the estimator alone. In this section you use a linear Gaussian latent-variable model, where the marginal likelihood and the posterior both have closed forms.

## The linear Gaussian latent-variable model

The latent variable has a standard normal prior in $d$ dimensions, and the observation in $D$ dimensions is an affine function of it plus isotropic Gaussian noise:

$$
\mathbf{Z} \sim \mathcal{N}(\mathbf{0}, I_d), \qquad \mathbf{X} \mid \mathbf{Z} = \mathbf{z} \sim \mathcal{N}(A\mathbf{z} + \mathbf{b},\ \sigma^2 I_D).
$$

Because $\mathbf{X} = A\mathbf{Z} + \mathbf{b} + \sigma \boldsymbol{\epsilon}$ is a linear function of Gaussian variables, the marginal is Gaussian too:

$$
p(\mathbf{x}) = \mathcal{N}\big(\mathbf{x};\ \mathbf{b},\ AA^\top + \sigma^2 I_D\big).
$$

The posterior over $\mathbf{z}$ is also Gaussian, $p(\mathbf{z} \mid \mathbf{x}) = \mathcal{N}(\mathbf{z};\ \mathbf{m}, S)$, with

$$
\begin{aligned}
S &= \left(I_d + \tfrac{1}{\sigma^2} A^\top A\right)^{-1}, \\
\mathbf{m} &= \tfrac{1}{\sigma^2}\, S A^\top (\mathbf{x} - \mathbf{b}).
\end{aligned}
$$

The observation dimension stays fixed at $D = 64$ and the noise level at $\sigma = 2$. The latent dimension $d$ varies. For each $d$ you draw a random matrix $A$ with standard normal entries, a random offset $\mathbf{b}$, and one observation $\mathbf{x}$ sampled from the model itself, so $\mathbf{x}$ is a typical point of the model.

In [ ]:
import numpy as np
from scipy.special import logsumexp
from scipy.stats import multivariate_normal

rng = np.random.default_rng(0)

D = 64  # observation dimension
SIGMA = 2.0  # observation noise standard deviation
N = 10_000  # samples per estimate
REPEATS = 20  # independent estimates per method and latent dimension


def make_model(d, rng):
    # Random linear Gaussian model and one observation drawn from it.
    A = rng.standard_normal((D, d))
    b = rng.standard_normal(D)
    z_true = rng.standard_normal(d)
    x = A @ z_true + b + SIGMA * rng.standard_normal(D)
    return A, b, x


def log_likelihood(x, Z, A, b):
    # log p(x | z) for each row z of Z.
    resid = x - (Z @ A.T + b)
    return -0.5 * np.sum(resid**2, axis=1) / SIGMA**2 - 0.5 * D * np.log(2 * np.pi * SIGMA**2)


def log_gaussian(Z, mean, chol):
    # log N(z; mean, chol @ chol.T) for each row z of Z.
    d = len(mean)
    Y = np.linalg.solve(chol, (Z - mean).T)
    return -0.5 * np.sum(Y**2, axis=0) - np.sum(np.log(np.diag(chol))) - 0.5 * d * np.log(2 * np.pi)


def log_prior(Z):
    return -0.5 * np.sum(Z**2, axis=1) - 0.5 * Z.shape[1] * np.log(2 * np.pi)


def exact_log_marginal(x, A, b):
    return multivariate_normal(mean=b, cov=A @ A.T + SIGMA**2 * np.eye(D)).logpdf(x)


def exact_posterior(x, A, b):
    d = A.shape[1]
    S = np.linalg.inv(np.eye(d) + A.T @ A / SIGMA**2)
    m = S @ A.T @ (x - b) / SIGMA**2
    return m, S

Two independent routes lead to the exact value of $\log p(\mathbf{x})$. The first evaluates the Gaussian marginal directly. The second uses Bayes' rule, which holds at every $\mathbf{z}$:

$$
\log p(\mathbf{x}) = \log p(\mathbf{x} \mid \mathbf{z}) + \log p(\mathbf{z}) - \log p(\mathbf{z} \mid \mathbf{x}).
$$

If the posterior formulas are right, the right-hand side does not depend on $\mathbf{z}$ and matches the marginal to rounding error. This holds at the posterior mean and at a random point.

In [ ]:
for d in (2, 10, 50):
    A, b, x = make_model(d, np.random.default_rng(d))
    m, S = exact_posterior(x, A, b)
    L = np.linalg.cholesky(S)
    Z = np.vstack([m, np.random.default_rng(1).standard_normal(d)])
    bayes = log_likelihood(x, Z, A, b) + log_prior(Z) - log_gaussian(Z, m, L)
    exact = exact_log_marginal(x, A, b)
    assert np.allclose(bayes, exact, rtol=0, atol=1e-8), (d, bayes, exact)
    print(f"d = {d:2d}: log p(x) = {exact:.6f}, Bayes identity gives {bayes[0]:.6f} and {bayes[1]:.6f}")

## Three estimators of the same integral

All three estimators below have the same form. Draw $\mathbf{z}_i$ from a proposal $q(\mathbf{z})$, compute the importance weight

$$
w_i = \frac{p(\mathbf{x} \mid \mathbf{z}_i)\, p(\mathbf{z}_i)}{q(\mathbf{z}_i)},
$$

and average, $\hat{p}(\mathbf{x}) = \frac{1}{N} \sum_i w_i$. The estimator is unbiased for any $q$ that is positive wherever the integrand is. Naive Monte Carlo is the special case $q = p(\mathbf{z})$, where $w_i = p(\mathbf{x} \mid \mathbf{z}_i)$. The three proposals are:

1. the prior $\mathcal{N}(\mathbf{0}, I_d)$, which is naive Monte Carlo;
2. a broadened posterior $\mathcal{N}(\mathbf{m}, c^2 S)$ with $c = 1.2$, a stand-in for a good but imperfect proposal;
3. the exact posterior $\mathcal{N}(\mathbf{m}, S)$.

With the exact posterior as proposal, the weight equals $p(\mathbf{x} \mid \mathbf{z}) p(\mathbf{z}) / p(\mathbf{z} \mid \mathbf{x}) = p(\mathbf{x})$ for every sample. Every sample returns the exact answer, and the estimator has zero variance.

The log-likelihoods here lie between about $-150$ and, for prior samples at $d = 50$, about $-990$ on average. A value such as $e^{-1000}$ underflows to zero in double precision. All computations therefore stay in the log domain, and the average uses the log-sum-exp identity:

$$
\log \hat{p}(\mathbf{x}) = \log \sum_{i=1}^N e^{\log w_i} - \log N.
$$

The *effective sample size* summarizes how evenly the weights are spread:

$$
\mathrm{ESS} = \frac{\left(\sum_i w_i\right)^2}{\sum_i w_i^2}.
$$

It equals $N$ when all weights are equal and approaches 1 when one sample carries all the weight. In log form it is $\exp\big(2\,\mathrm{LSE}(\log w) - \mathrm{LSE}(2 \log w)\big)$.

In [ ]:
def sample_proposal(method, m, S, rng):
    # Return samples Z and log q(Z) for the named proposal.
    d = len(m)
    eps = rng.standard_normal((N, d))
    if method == "prior":
        return eps, log_prior(eps)
    c = {"broadened": 1.2, "posterior": 1.0}[method]
    chol = c * np.linalg.cholesky(S)
    Z = m + eps @ chol.T
    return Z, log_gaussian(Z, m, chol)


def log_weights(method, x, A, b, m, S, rng):
    Z, log_q = sample_proposal(method, m, S, rng)
    return log_likelihood(x, Z, A, b) + log_prior(Z) - log_q


def log_marginal_estimate(log_w):
    return logsumexp(log_w) - np.log(len(log_w))


def effective_sample_size(log_w):
    return np.exp(2 * logsumexp(log_w) - logsumexp(2 * log_w))

## Where the prior samples land

In two latent dimensions you can see the problem directly. The figure shows 2,000 prior samples. Each dot's size and color give its share of the total weight. The dashed circle is two standard deviations of the prior, and the solid ellipse is two standard deviations of the exact posterior.

In [ ]:
import matplotlib.pyplot as plt

A2, b2, x2 = make_model(2, np.random.default_rng(2))
m2, S2 = exact_posterior(x2, A2, b2)
Z_show = np.random.default_rng(3).standard_normal((2000, 2))
lw_show = log_likelihood(x2, Z_show, A2, b2)
share = np.exp(lw_show - logsumexp(lw_show))

theta = np.linspace(0, 2 * np.pi, 200)
circle = np.stack([np.cos(theta), np.sin(theta)])
post_ellipse = m2[:, None] + 2 * np.linalg.cholesky(S2) @ circle

fig, ax = plt.subplots(figsize=(5.5, 5.5))
order = np.argsort(share)
sc = ax.scatter(Z_show[order, 0], Z_show[order, 1], s=3 + 60 * share[order] / share.max(),
                c=share[order], cmap="viridis", alpha=0.8)
ax.plot(2 * circle[0], 2 * circle[1], "k--", lw=1, label="prior, 2 std")
ax.plot(post_ellipse[0], post_ellipse[1], "r-", lw=1.5, label="posterior, 2 std", zorder=3)
ax.set_aspect("equal")
ax.set_xlabel("$z_1$")
ax.set_ylabel("$z_2$")
ax.legend(loc="upper right")
fig.colorbar(sc, ax=ax, shrink=0.8, label="share of total weight")
ax.set_title(f"d = 2: ESS = {effective_sample_size(lw_show):.0f} of {len(Z_show)} prior samples")
plt.tight_layout()
plt.show()

Only the samples inside or near the posterior ellipse carry weight; the rest contribute almost nothing to the sum. In two dimensions this still leaves about a tenth of the samples doing useful work. The fraction of prior mass that falls inside the posterior region shrinks roughly geometrically with each added latent dimension, because each dimension contributes its own factor.

## Sweeping the latent dimension

The sweep below runs each estimator 20 times with $N = 10{,}000$ samples, at latent dimensions from 1 to 50. For each $d$ it reports the error $\log \hat{p}(\mathbf{x}) - \log p(\mathbf{x})$ in nats and the median effective sample size.

In [ ]:
METHODS = ("prior", "broadened", "posterior")
D_GRID = (1, 2, 3, 4, 6, 8, 10, 15, 20, 30, 40, 50)

results = {}
for d in D_GRID:
    A, b, x = make_model(d, rng)
    m, S = exact_posterior(x, A, b)
    exact = exact_log_marginal(x, A, b)
    for method in METHODS:
        errors, ess = [], []
        for _ in range(REPEATS):
            log_w = log_weights(method, x, A, b, m, S, rng)
            errors.append(log_marginal_estimate(log_w) - exact)
            ess.append(effective_sample_size(log_w))
        results[d, method] = {"exact": exact, "errors": np.array(errors), "ess": np.array(ess), "log_w": log_w}

print(f"{'d':>3} {'proposal':>10} {'log p(x)':>10} {'mean error':>11} {'std error':>10} {'median ESS':>11}")
for d in (2, 10, 50):
    for method in METHODS:
        r = results[d, method]
        print(f"{d:>3} {method:>10} {r['exact']:>10.2f} {r['errors'].mean():>11.3f} "
              f"{r['errors'].std():>10.3f} {np.median(r['ess']):>11.1f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
d_arr = np.array(D_GRID)
styles = {"prior": ("C3", "o", "prior (naive MC)"),
          "broadened": ("C0", "s", "broadened posterior, c = 1.2"),
          "posterior": ("C2", "^", "exact posterior")}
for method in METHODS:
    color, marker, label = styles[method]
    ess = np.array([np.median(results[d, method]["ess"]) for d in D_GRID])
    err = np.array([results[d, method]["errors"].mean() for d in D_GRID])
    sd = np.array([results[d, method]["errors"].std() for d in D_GRID])
    axes[0].plot(d_arr, ess, color=color, marker=marker, label=label)
    axes[1].plot(d_arr, err, color=color, marker=marker, label=label)
    axes[1].fill_between(d_arr, err - sd, err + sd, color=color, alpha=0.2)
axes[0].set_yscale("log")
axes[0].set_xlabel("latent dimension d")
axes[0].set_ylabel(f"median ESS (N = {N})")
axes[0].set_title("Effective sample size")
axes[1].set_yscale("symlog", linthresh=1)
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_xlabel("latent dimension d")
axes[1].set_ylabel("estimate minus exact log p(x) (nats)")
axes[1].set_title("Log-likelihood error, mean and std over 20 runs")
axes[0].legend()
plt.tight_layout()
plt.show()

At $d = 2$ naive Monte Carlo is accurate to a few hundredths of a nat, with several hundred effective samples. By $d = 10$ the median effective sample size is about 2, so one or two prior samples decide each estimate. The estimate is then about a nat too low on average, and single runs scatter by more than a nat. For larger $d$, the effective sample size falls to 1 and the error grows steadily, to more than 200 nats at $d = 50$. That is a factor of more than $e^{200}$ in $p(\mathbf{x})$. The error is negative because the log of an unbiased estimator is biased downward. The weights are heavy-tailed, and the rare large weights that would correct the average almost never appear in a sample of $10{,}000$.

The broadened posterior stays accurate at every dimension, but its effective sample size also decays with $d$. For a proposal $\mathcal{N}(\mathbf{m}, c^2 S)$ and target $\mathcal{N}(\mathbf{m}, S)$, the ratio $N / \mathrm{ESS}$ approaches $\big(c^2 / \sqrt{2c^2 - 1}\big)^d$ for large $N$. With $c = 1.2$ the base is about 1.05, so the loss is slow, but it is still exponential in $d$. A proposal that has the right location and the right shape up to a modest scale error already costs about 90 percent of the samples at $d = 50$. The exact posterior keeps the effective sample size at $N$ for every $d$.

The exact-posterior proposal returns $\log p(\mathbf{x})$ from every single sample, to $10^{-8}$ nats, with an effective sample size equal to $N$. The broadened proposal stays within 0.2 nats in every run. Naive Monte Carlo is within 0.1 nats on average at $d = 2$, and misses by more than 20 nats with an effective sample size below 2 at $d = 50$.

In [ ]:
for d in D_GRID:
    post = results[d, "posterior"]
    assert np.allclose(post["log_w"], post["exact"], rtol=0, atol=1e-8), d
    assert np.allclose(post["ess"], N, rtol=1e-9, atol=0), (d, post["ess"])
    assert np.all(np.abs(results[d, "broadened"]["errors"]) < 0.2), d

naive2, naive50 = results[2, "prior"], results[50, "prior"]
assert abs(naive2["errors"].mean()) < 0.1, naive2["errors"].mean()
assert naive50["errors"].mean() < -20, naive50["errors"].mean()
assert np.median(naive50["ess"]) < 2, np.median(naive50["ess"])

c = 1.2
for d in (10, 50):
    predicted = N / (c**2 / np.sqrt(2 * c**2 - 1)) ** d
    observed = np.median(results[d, "broadened"]["ess"])
    print(f"d = {d}: broadened-proposal ESS {observed:.0f}, large-N prediction {predicted:.0f}")
print("All estimator checks passed.")

## The ELBO and its gap

Importance sampling needs a good proposal, and the best one is the posterior. Variational inference turns the same idea into an optimization problem. For any density $q(\mathbf{z})$, the *evidence lower bound* is

$$
\mathrm{ELBO}(q) = \mathbb{E}_{q}\big[\log p(\mathbf{x} \mid \mathbf{z}) + \log p(\mathbf{z}) - \log q(\mathbf{z})\big],
$$

which is the expected log importance weight under $q$. It differs from the log marginal likelihood by exactly one term [@Blei2016-ui]:

$$
\log p(\mathbf{x}) = \mathrm{ELBO}(q) + \mathrm{KL}\big(q(\mathbf{z}) \,\|\, p(\mathbf{z} \mid \mathbf{x})\big).
$$

Since the KL divergence is non-negative, the ELBO is a lower bound, and the gap measures how far $q$ is from the posterior. In the linear Gaussian model both sides have closed forms for any Gaussian $q = \mathcal{N}(\boldsymbol{\mu}, V)$, so you can check the identity term by term. Three choices of $q$ are compared:

1. the prior $\mathcal{N}(\mathbf{0}, I_d)$;
2. the mean-field Gaussian $\mathcal{N}(\mathbf{m}, \operatorname{diag}(v))$ with $v_j = 1 / (S^{-1})_{jj}$, which is the diagonal Gaussian that minimizes $\mathrm{KL}(q \,\|\, p(\mathbf{z} \mid \mathbf{x}))$ here;
3. the exact posterior, where the gap must be zero.

The ELBO uses the expected log-likelihood

$$
\begin{aligned}
\mathbb{E}_q[\log p(\mathbf{x} \mid \mathbf{z})] &= -\tfrac{D}{2}\log(2\pi\sigma^2) \\
&\quad - \tfrac{1}{2\sigma^2}\big(\lVert \mathbf{x} - A\boldsymbol{\mu} - \mathbf{b} \rVert^2 + \operatorname{tr}(A^\top A V)\big),
\end{aligned}
$$

and the KL divergence between two Gaussians in closed form. A Monte Carlo average of $\log w$ under $q$ gives an independent estimate of the same ELBO.

In [ ]:
def gaussian_kl(mu0, V0, mu1, V1):
    # KL(N(mu0, V0) || N(mu1, V1)).
    d = len(mu0)
    V1_inv = np.linalg.inv(V1)
    diff = mu1 - mu0
    _, logdet0 = np.linalg.slogdet(V0)
    _, logdet1 = np.linalg.slogdet(V1)
    return 0.5 * (np.trace(V1_inv @ V0) + diff @ V1_inv @ diff - d + logdet1 - logdet0)


def elbo_closed_form(x, A, b, mu, V):
    d = len(mu)
    resid = x - A @ mu - b
    exp_loglik = -0.5 * D * np.log(2 * np.pi * SIGMA**2) - (resid @ resid + np.trace(A.T @ A @ V)) / (2 * SIGMA**2)
    exp_logprior = -0.5 * d * np.log(2 * np.pi) - 0.5 * (mu @ mu + np.trace(V))
    _, logdetV = np.linalg.slogdet(V)
    entropy = 0.5 * d * np.log(2 * np.pi * np.e) + 0.5 * logdetV
    return exp_loglik + exp_logprior + entropy


def elbo_monte_carlo(x, A, b, mu, V, rng):
    chol = np.linalg.cholesky(V)
    Z = mu + rng.standard_normal((N, len(mu))) @ chol.T
    log_w = log_likelihood(x, Z, A, b) + log_prior(Z) - log_gaussian(Z, mu, chol)
    return log_w.mean(), log_w.std() / np.sqrt(N)


elbo_rows = []
print(f"{'d':>3} {'q':>11} {'ELBO':>10} {'MC ELBO':>10} {'gap':>9} {'KL(q||post)':>12}")
for d in (2, 10, 50):
    A, b, x = make_model(d, rng)
    m, S = exact_posterior(x, A, b)
    exact = exact_log_marginal(x, A, b)
    choices = {"prior": (np.zeros(d), np.eye(d)),
               "mean-field": (m, np.diag(1 / np.diag(np.linalg.inv(S)))),
               "posterior": (m, S)}
    for name, (mu, V) in choices.items():
        elbo = elbo_closed_form(x, A, b, mu, V)
        mc, se = elbo_monte_carlo(x, A, b, mu, V, rng)
        kl = gaussian_kl(mu, V, m, S)
        elbo_rows.append((d, name, exact, elbo, mc, se, kl))
        print(f"{d:>3} {name:>11} {elbo:>10.3f} {mc:>10.3f} {exact - elbo:>9.3f} {kl:>12.3f}")

In every row the gap $\log p(\mathbf{x}) - \mathrm{ELBO}$ equals the KL divergence from $q$ to the posterior. With the prior as $q$, the gap grows from tens of nats at $d = 2$ to hundreds at $d = 50$. This is the same mismatch that made naive Monte Carlo fail. The mean-field Gaussian has the right mean and the right conditional spread in each coordinate. Its gap is a fraction of a nat at $d = 2$ and $d = 10$, and about 11 nats at $d = 50$. The remainder comes from correlations between latent coordinates, which a diagonal covariance cannot represent. With the exact posterior the bound is tight. The identity holds to $10^{-8}$ nats in every row. Each Monte Carlo ELBO lies within four standard errors of its closed form, and the gap vanishes for the exact posterior.

In [ ]:
for d, name, exact, elbo, mc, se, kl in elbo_rows:
    assert kl >= -1e-10, (d, name, kl)
    assert abs(elbo + kl - exact) < 1e-8, (d, name, elbo + kl - exact)
    assert abs(mc - elbo) < 4 * se + 1e-8, (d, name, mc, elbo, se)
    if name == "posterior":
        assert abs(exact - elbo) < 1e-8, (d, exact - elbo)
print("All ELBO checks passed.")

## From the exact posterior to a learned one

In the linear Gaussian model the posterior $p(\mathbf{z} \mid \mathbf{x})$ is available in closed form, and it solves both problems on this page at once. As a proposal it turns importance sampling into an exact computation, and as the variational distribution it closes the ELBO gap. For a model whose likelihood is a neural network decoder, the posterior has no closed form, and the prior is a useless proposal for the same reason it fails here: the latent region that explains a given $\mathbf{x}$ is a tiny fraction of the prior's mass.

A variational autoencoder replaces the exact posterior with an approximate posterior $q_\phi(\mathbf{z} \mid \mathbf{x})$, a Gaussian whose mean and covariance come from an encoder network evaluated at $\mathbf{x}$ [@Kingma2013-gm]. Training maximizes the ELBO, which by the identity above pushes $q_\phi(\mathbf{z} \mid \mathbf{x})$ toward the true posterior while it improves the decoder. The trained encoder can also serve as the importance-sampling proposal when you want an estimate of $\log p(\mathbf{x})$ itself. The [VAE architecture](/aiml-common/lectures/vae/vae-architecture/index) section builds this encoder and decoder pair.